In [2]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
jan_2026_dl_gen_ai_project_path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print('Data source import complete.')



100%|██████████| 16.1G/16.1G [02:57<00:00, 97.3MB/s]

Extracting files...


Data source import complete.


# Q.1

In [ ]:
from sklearn.model_selection import train_test_split

genres = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]
all_recipes = []
for genre in genres:
    for i in range(100):
        recipe = {
            "genre": genre,
            "file_index": i,
        }
        all_recipes.append(recipe)

print(f"Total recipes: {len(all_recipes)}")  
train_recipes, val_recipes = train_test_split(
    all_recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print(f"Train recipes: {len(train_recipes)}") 
print(f"Val recipes:   {len(val_recipes)}")    

Total recipes: 1000
Train recipes: 800
Val recipes:   200


# Q.2

In [26]:
import numpy as np
import librosa

sr=16000 
duration=10
target_len=sr*duration

stems = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]
song_path='/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00069'
noise_path='/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/3-216281-A-39.wav'

audio_mix=None
for stem in stems:
    stem_path = os.path.join(song_path, stem)
    audio,_=librosa.load(stem_path, sr=sr, duration=duration)
    if len(audio)<target_len:
        audio=np.pad(audio,(0,target_len-len(audio)))
    else:
        audio=audio[:target_len]
    
    if audio_mix is None:
        audio_mix=audio
    else:
        audio_mix+=audio

noise,_=librosa.load(noise_path, sr=sr, duration=duration)
if len(noise)<target_len:
    noise=np.pad(noise,(0,target_len-len(noise)))       
else:
    noise=noise[:target_len]
noise=noise*0.2

final_mix=audio_mix+noise

print(f"Shape after mixing : {final_mix.shape}")


Shape after mixing : (160000,)


# Q.3

In [21]:
import warnings
warnings.filterwarnings("ignore")

In [22]:
import torch
from transformers import AutoFeatureExtractor

extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

mix = np.ones(160000)

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs["input_values"].squeeze(0)

print(f"Shape after squeeze: {input_values.shape}")

Shape after squeeze: torch.Size([1024, 128])


# Q.4

In [23]:
from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {trainable_params}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 86196490


# Q.5

In [25]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])
y_normalized = y_test / (np.max(np.abs(y_test)) + 1e-9)
print(f"Index 0 value:    {y_normalized[0]:.3f}")

Index 0 value:    -1.000
